In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/Datasets/PercolationRate', exist_ok=True)

In [ ]:
# @title 1.1 Installation des dépendances système
!apt-get update -qq
!apt-get install -y -qq build-essential cmake git libeigen3-dev libomp-dev libtbb-dev libtbbmalloc2

# libboost-all-dev est souvent nécessaire pour que CMake détecte correctement CGAL
!apt-get install -y -qq libcgal-dev libgmp-dev libmpfr-dev libboost-all-dev

In [ ]:
# @title 1.2 Installation des dépendances Python
!pip install -q --upgrade pip setuptools wheel Cython cmake jedi gdown pybind11
!pip install -q numpy scipy scikit-learn plotly tqdm joblib open3d plyfile hdbscan pandas matplotlib pyyaml shapely mip POT


In [ ]:
%%bash
# @title 1.3 Installation de HGP-clusterer
set -euo pipefail
WORKDIR="/content"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

# HGP-clusterer
if [ -d HGP-clusterer ]; then
    git -C HGP-clusterer pull --ff-only
else
    git clone https://github.com/Ludwig-H/HGP-clusterer.git
fi



In [ ]:
from tqdm.auto import tqdm
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
# @title 1.4 Compilation de HGP
import os
import sys
import subprocess

WORKDIR = "/content"
os.chdir(WORKDIR)

print("Configuration CGAL...")

# -- FIX: Add CGAL path to environment for setup_cgal.py --
cgal_prefix = "/usr/lib/x86_64-linux-gnu/cmake/CGAL"
current_cpp = os.environ.get("CMAKE_PREFIX_PATH", "")
os.environ["CMAKE_PREFIX_PATH"] = f"{current_cpp}:{cgal_prefix}" if current_cpp else cgal_prefix
# ---------------------------------------------------------

# On tente de construire l'outil CGAL, mais on continue même en cas d'erreur
# car le setup.py principal pourrait réussir autrement.
try:
    subprocess.run(["python3", f"{WORKDIR}/HGP-clusterer/scripts/setup_cgal.py"], check=True)
except subprocess.CalledProcessError:
    tqdm.write("⚠️ Attention: Echec du script setup_cgal.py. Tentative de continuation avec le build principal...")

os.chdir(f"{WORKDIR}/HGP-clusterer")
!rm -rf build dist *.egg-info

install_cmd = "pip install --no-build-isolation -v --no-deps ."
# Ajout du chemin système pour CGAL (Debian/Ubuntu/Colab)
# Note: On le passe aussi explicitement ici pour être sûr
install_cmd = f"CGALDELAUNAY_ROOT={WORKDIR}/HGP-clusterer/CGALDelaunay CMAKE_PREFIX_PATH={WORKDIR}/HGP-clusterer:{cgal_prefix} {install_cmd}"

print(f"Exécution : {install_cmd}")
!{install_cmd}

os.environ["CGALDELAUNAY_ROOT"] = f"{WORKDIR}/HGP-clusterer/CGALDelaunay"

try:
    from hgp_delaunay import HGPDelaunay
    tqdm.write("✅ HGPDelaunay installé.")
except ImportError as e:
    tqdm.write(f"❌ Erreur import HGP: {e}")


In [ ]:
import numpy as np
import scipy.sparse as sp
import os
import pickle
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.neighbors import NearestNeighbors
from hgp_delaunay import HGPDelaunay
from numba import njit

In [ ]:
@njit
def simulate_robust_dbscan(times, types, nodes_u, nodes_v, N):
    parent = np.arange(N)
    rank = np.zeros(N, dtype=np.int32)
    size_rob = np.zeros(N, dtype=np.int32)
    size_dbs = np.zeros(N, dtype=np.int32)

    ans_rob = np.full(N + 1, np.nan)
    ans_dbs = np.full(N + 1, np.nan)
    max_rob = 0
    max_dbs = 0

    for i in range(len(times)):
        t = times[i]
        typ = types[i]
        u = nodes_u[i]

        curr = u
        while parent[curr] != curr:
            curr = parent[curr]
        ru = curr
        curr = u
        while curr != ru:
            nxt = parent[curr]
            parent[curr] = ru
            curr = nxt

        if typ == 0:
            size_dbs[ru] += 1
            if size_dbs[ru] > max_dbs:
                for _idx in range(max_dbs + 1, size_dbs[ru] + 1):
                    ans_dbs[_idx] = t
                max_dbs = size_dbs[ru]
        elif typ == 1:
            size_rob[ru] += 1
            if size_rob[ru] > max_rob:
                for _idx in range(max_rob + 1, size_rob[ru] + 1):
                    ans_rob[_idx] = t
                max_rob = size_rob[ru]
        else:
            v = nodes_v[i]
            curr = v
            while parent[curr] != curr:
                curr = parent[curr]
            rv = curr
            curr = v
            while curr != rv:
                nxt = parent[curr]
                parent[curr] = rv
                curr = nxt

            if ru != rv:
                if rank[ru] < rank[rv]:
                    tmp = ru
                    ru = rv
                    rv = tmp
                parent[rv] = ru
                if rank[ru] == rank[rv]:
                    rank[ru] += 1

                size_rob[ru] += size_rob[rv]
                size_dbs[ru] += size_dbs[rv]

                if size_rob[ru] > max_rob:
                    for _idx in range(max_rob + 1, size_rob[ru] + 1):
                        ans_rob[_idx] = t
                    max_rob = size_rob[ru]
                if size_dbs[ru] > max_dbs:
                    for _idx in range(max_dbs + 1, size_dbs[ru] + 1):
                        ans_dbs[_idx] = t
                    max_dbs = size_dbs[ru]

    return ans_rob[1:], ans_dbs[1:]

@njit
def compute_wu(real_u, real_v, del_w, r_core, w_u, target_u):
    for i in range(len(real_u)):
        u = real_u[i]
        v = real_v[i]
        w = del_w[i]

        val_v = max(r_core[v], w)
        if val_v < w_u[u]:
            w_u[u] = val_v
            target_u[u] = v

        val_u = max(r_core[u], w)
        if val_u < w_u[v]:
            w_u[v] = val_u
            target_u[v] = u

def compute_hgp_percolation(faces_unique, e_u, e_v, e_w, n_points):
    F = faces_unique.shape[0]
    order = np.argsort(e_w)
    e_u = e_u[order]
    e_v = e_v[order]
    e_w = e_w[order]

    parent = np.arange(F)
    def find(i):
        path = []
        while parent[i] != i:
            path.append(i)
            i = parent[i]
        for node in path:
            parent[node] = i
        return i

    components = [set(faces_unique[i]) for i in range(F)]
    max_size = len(faces_unique[0]) if F > 0 else 0
    ans = np.full(n_points + 1, np.nan)
    ans[1:max_size + 1] = 0.0

    for i in range(len(e_w)):
        u = e_u[i]
        v = e_v[i]
        w = e_w[i]

        ru = find(u)
        rv = find(v)

        if ru != rv:
            if len(components[ru]) < len(components[rv]):
                ru, rv = rv, ru
            components[ru].update(components[rv])
            components[rv] = None
            parent[rv] = ru

            new_size = len(components[ru])
            if new_size > max_size:
                ans[max_size + 1 : new_size + 1] = w
                max_size = new_size

    return ans[1:]

In [ ]:
# @title Parameters
import os
import pickle
K_list = [1, 2, 3, 4, 5] # @param
d = 2 # @param
n = 1000000 # @param
T = 1 # @param

save_path = f'/content/drive/MyDrive/Datasets/PercolationRate/percolation_results_d{d}_n{n}.pkl'
print(f"Results will be saved to/loaded from: {save_path}")


In [ ]:
# @title Run Simulations
from tqdm.auto import tqdm

if os.path.exists(save_path):
    with open(save_path, 'rb') as f:
        results = pickle.load(f)
    tqdm.write(f"Loaded existing results.")
else:
    results = {'hgp': {k: [] for k in K_list},
               'robust': {k: [] for k in K_list},
               'dbscan': {k: [] for k in K_list}}

tests_done = min([len(results['hgp'].get(k, [])) for k in K_list]) if K_list else 0
target_tests = tests_done + T
print(f"{tests_done} tests already done. Adding {T} new tests, targeting {target_tests} total.")

for t_idx in tqdm(range(tests_done, target_tests), desc="Tests"):
    tqdm.write(f"\n--- Starting test {t_idx+1}/{target_tests} ---")
    X = np.random.uniform(0, n**(1/d), size=(n, d)).astype(np.float32)

    # Get edges for Robust and DBSCAN from 20-NN graph (skipping HGP 1-skeleton for speed)
    tqdm.write("Computing 20-NN graph for Robust/DBSCAN...")
    tree = cKDTree(X)
    
    dists20, inds20 = tree.query(X, k=21, workers=-1)
    real_u = np.repeat(np.arange(n, dtype=np.int32), 20)
    real_v = inds20[:, 1:].flatten().astype(np.int32)
    del_w = (dists20[:, 1:].flatten() / 2.0).astype(np.float32)

    for K in K_list:
        tqdm.write(f"  > Processing K={K}...")
        if K not in results['hgp']:
            results['hgp'][K] = []
            results['robust'][K] = []
            results['dbscan'][K] = []

        if len(results['hgp'][K]) >= target_tests:
            tqdm.write(f"  > Skipping K={K}, already has {len(results['hgp'][K])} tests.")
            continue

        # 1. HGP Components
        tqdm.write("    Computing HGP-Delaunay graph...")
        hgp = HGPDelaunay(backend='cgal', K=K, expZ=1.0, verbose=True)
        faces, eu, ev, ew = hgp.fit_transform(X)
        ans_hgp = compute_hgp_percolation(faces, eu, ev, ew, n)
        results['hgp'][K].append(ans_hgp)

        # 2. Nearest Neighbors for Core Distance
        tqdm.write("    Computing Core Distances (k-NN)...")
        # Reusing precomputed dists20
        
        
        r_core = (dists20[:, K] / 2.0).astype(np.float32)

        # 3. Simulate DBSCAN and Robust Single-Linkage
        tqdm.write("    -> Computing wu (Numba JIT may take a few seconds)...")
        w_u = r_core.copy()
        target_u = np.arange(n, dtype=np.int32)
        compute_wu(real_u, real_v, del_w, r_core, w_u, target_u)

        tqdm.write("    -> Computing w_cc and edge arrays...")
        w_cc = np.maximum(r_core[real_u], r_core[real_v])
        w_cc = np.maximum(w_cc, del_w)

        E = len(real_u)
        times = np.concatenate([w_u, r_core, w_cc])
        types = np.concatenate([np.zeros(n, dtype=np.int8),
                                np.ones(n, dtype=np.int8),
                                np.full(E, 2, dtype=np.int8)])
        nodes_u = np.concatenate([target_u, np.arange(n, dtype=np.int32), real_u])
        nodes_v = np.concatenate([np.zeros(n, dtype=np.int32), np.zeros(n, dtype=np.int32), real_v])

        tqdm.write(f"    -> Sorting ~{len(times)/1e6:.1f} million edges (takes a few seconds)...")
        order = np.argsort(times)
        times = times[order]
        types = types[order]
        nodes_u = nodes_u[order]
        nodes_v = nodes_v[order]
        del order
        import gc; gc.collect()

        tqdm.write("    Simulating Robust Single-Linkage and DBSCAN-core...")
        ans_rob, ans_dbs = simulate_robust_dbscan(times, types, nodes_u, nodes_v, n)
        tqdm.write('    -> Simulation finished!')
        import sys; sys.stdout.flush()
        results['robust'][K].append(ans_rob)
        results['dbscan'][K].append(ans_dbs)
        
        # Aggressive memory cleanup
        del faces, eu, ev, ew, hgp, ans_hgp
        del w_u, target_u, w_cc, times, types, nodes_u, nodes_v
        del ans_rob, ans_dbs
        import gc; gc.collect()

    
    # Cleanup end of test
    del X, real_u, real_v, del_w
    del tree, dists20, inds20
    import gc; gc.collect()
    
    with open(save_path, 'wb') as f:
        pickle.dump(results, f)

In [ ]:
# @title Analysis and Plotting
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
from tqdm.auto import tqdm

if os.path.exists(save_path):
    with open(save_path, 'rb') as f:
        results_plot = pickle.load(f)
else:
    print("No results found to plot!")
    results_plot = {'hgp': {}, 'robust': {}, 'dbscan': {}}

tests_count = min([len(results_plot['hgp'].get(k, [])) for k in K_list if k in results_plot['hgp']] + [0]) if K_list else 0
print(f"\n--- Plotting results based on {tests_count} tests ---\n")

epsilon = 0.03
n_min = int(n * epsilon)
n_max = int(n * (1 - epsilon))

v_results = {'hgp': {}, 'robust': {}, 'dbscan': {}}

for K in K_list:
    if K not in results_plot['hgp'] or len(results_plot['hgp'][K]) == 0: continue

    avg_hgp = np.nanmean(np.array(results_plot['hgp'][K]), axis=0)
    avg_rob = np.nanmean(np.array(results_plot['robust'][K]), axis=0)
    avg_dbs = np.nanmean(np.array(results_plot['dbscan'][K]), axis=0)

    r_min_hgp = avg_hgp[n_min]
    r_max_hgp = avg_hgp[n_max]
    v_results['hgp'][K] = (r_min_hgp / r_max_hgp)**d if r_max_hgp > 0 else 0

    r_min_rob = avg_rob[n_min]
    r_max_rob = avg_rob[n_max]
    v_results['robust'][K] = (r_min_rob / r_max_rob)**d if r_max_rob > 0 else 0

    r_min_dbs = avg_dbs[n_min]
    r_max_dbs = avg_dbs[n_max]
    v_results['dbscan'][K] = (r_min_dbs / r_max_dbs)**d if r_max_dbs > 0 else 0

print(f"=== Percolation Rate v (d={d}) ===")
print(f"{'K':<5} | {'HGP':<10} | {'Robust SL':<10} | {'DBSCAN-core':<10}")
print("-" * 45)
for K in K_list:
    if K not in v_results['hgp']: continue
    tqdm.write(f"{K:<5} | {v_results['hgp'][K]:<10.4f} | {v_results['robust'][K]:<10.4f} | {v_results['dbscan'][K]:<10.4f}")

def plot_curves(res_dict, name):
    plt.figure(figsize=(10, 6))
    x_axis = np.arange(1, n+1) / n
    for K in K_list:
        if K not in res_dict: continue
        avg_curve = np.nanmean(np.array(res_dict[K]), axis=0)
        intensity = (2 * avg_curve)**d
        plt.plot(intensity, x_axis, label=f'{name} K={K}')
    plt.legend()
    plt.grid(True)
    plt.xlim(0, 10)
    plt.show()

plot_curves(results_plot['hgp'], 'HGP')
plot_curves(results_plot['robust'], 'Robust SL')
plot_curves(results_plot['dbscan'], 'DBSCAN')